# Phase 4 (CJPE) - Model 1 (GQA) + LoRMA

Court judgment prediction (binary, macro-F1). Phase-1 backbone is **frozen**; only the hand-written LoRMA layers and a small hierarchical head are trained. No `peft` library.

**Input of one judgment:** up to `MAX_CHUNKS` chunks of `CHUNK_LEN` tokens (first chunk + the last chunks, because the end of a judgment carries most of the signal in the data analysis).
**Model:** backbone -> per-chunk vector (mean of token states + state at the final SEP) -> BiGRU over chunks -> attention pooling -> 2 logits.

**Attach as inputs:**
1. Output of the CJPE data-prep notebook (`tokenized_cjpe.pkl`).
2. Output of your Phase-1 notebook with the weights: folder `legal-gqa-final`.
3. `legal_gqa_model.py` (for example the output of your Phase-2 Model-1 notebook, which wrote it).

**Steps:** run once with `SMOKE_TEST = True` (a few minutes), then set it to `False` for the real run.

**Outputs** (`/kaggle/working/cjpe_model1_lorma/`): `submission.zip` and `prediction.json` (upload to Codabench), `results.json`, `val_predictions.csv`, `val_logits.npy`, `test_logits.npy`, `poolb_logits.npy` (used by the KD notebook), `best.pt`.

If the session stops early (time cap), it saves `ckpt.pt`. Add this notebook's output as an input of the next run and run again: it resumes by itself.

## 1. Settings

In [ ]:
# ------------------------------------------------------------------ this run
EXPERIMENT_NAME = "cjpe_model1_lorma"
MODEL_KIND = "model1"       # model1 = GQA, model4 = GQA+MHA+MoE
ADAPTER = "lorma"             # lora | lorma | adapter
SMOKE_TEST = True                   # True = tiny quick run to check that everything works. Set False for the real run.

# ------------------------------------------------------------------ inputs (None = search /kaggle/input)
DATA_PKL = None
MODEL_DIR = None
MODEL_SRC = None
RESUME_DIR = None
OUT_DIR = "/kaggle/working/" + EXPERIMENT_NAME

# ------------------------------------------------------------------ data
SEED = 42
N_TRAIN_DOCS = 10000        # pool A: documents used to train the adapters (same pool in every notebook)
N_POOLB_DOCS = 10000        # pool B: unseen train documents, used later as the knowledge-distillation transfer set
SAVE_POOLB_LOGITS = True    # predict pool B at the end so this run can be a KD teacher (costs some extra minutes)
VIEW_MODE = "first_last"    # first_last | last | first | uniform  (which chunks are kept if a judgment is too long)
CHUNK_LEN = 1024            # tokens per chunk, including the final SEP (must be <= Phase-1 context length)
CHUNK_OVERLAP = 128
MAX_CHUNKS = 4
N_FIRST_CHUNKS = 1          # used by first_last: how many chunks come from the beginning

# ------------------------------------------------------------------ head
DOC_HEAD = "bigru"          # bigru | attn  (how chunk vectors are combined)
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.1

# ------------------------------------------------------------------ LoRA (hand-written)
LORA_R = 16
LORA_ALPHA = 32             # scale = alpha / r
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ------------------------------------------------------------------ LoRMA (hand-written)
LORMA_VARIANT = "plus"      # plus | perm   (perm did not learn on Model 1 in our run: val AUC 0.47, see report)
PERM_INIT = "paper"         # paper | lora_style  (only for perm; see the note in the layer)
LORMA_R = 16
LORMA_ALPHA = 16
LORMA_DROPOUT = 0.05
LORMA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ------------------------------------------------------------------ bottleneck adapters (hand-written)
ADAPTER_BOTTLENECK = 64
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["o_proj", "down_proj"]   # an adapter after the attention output and after the FFN output

# ------------------------------------------------------------------ training
EPOCHS = 2
BATCH_DOCS = 16             # documents per optimizer step
MICRO_DOCS = 4              # documents per forward pass (lower it if the GPU runs out of memory)
EVAL_MICRO_DOCS = 8
LR_ADAPTER = 5e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01         # head only
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.0
CLASS_WEIGHT_POWER = 0.5    # weights = (1 / class_count) ** power, normalised. 0.5 = inverse-sqrt, 1.0 = inverse, 0 = none
USE_CHECKPOINT = True       # activation checkpointing: slower, much less GPU memory
EVAL_EVERY_STEPS = 300
SAVE_EVERY_STEPS = 100
PATIENCE = 3                # stop if the validation score does not improve for this many evaluations
MAX_SESSION_HOURS = 10.0    # pause and save before the Kaggle session limit
STOP_AFTER_STEPS = None     # test hook: pause after this many steps in this session
LOG_EVERY_STEPS = 20
VAL_MAX_DOCS = None
TEST_MAX_DOCS = None

if SMOKE_TEST:
    N_TRAIN_DOCS = 48
    N_POOLB_DOCS = 24
    VAL_MAX_DOCS = 40
    TEST_MAX_DOCS = 20
    EPOCHS = 1
    CHUNK_LEN = 256
    CHUNK_OVERLAP = 32
    MAX_CHUNKS = 2
    BATCH_DOCS = 8
    MICRO_DOCS = 2
    EVAL_EVERY_STEPS = 3
    SAVE_EVERY_STEPS = 2
    LOG_EVERY_STEPS = 1

# test hook: lets a local test change settings without editing the notebook (unused on Kaggle)
import os
_overrides = os.environ.get("CJPE_OVERRIDES", "")
if _overrides != "":
    exec(_overrides)
    print("overrides applied:", _overrides)

if SMOKE_TEST:
    print("*" * 70)
    print("SMOKE TEST is ON: tiny run, results are meaningless. Set SMOKE_TEST = False for the real run.")
    print("*" * 70)

## 2. Imports and helpers

In [ ]:
import re
import sys
import json
import glob
import time
import math
import pickle
import random
import shutil
import hashlib
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint

assert ADAPTER in ("lora", "lorma", "adapter")
assert VIEW_MODE in ("first_last", "last", "first", "uniform")
assert DOC_HEAD in ("bigru", "attn")
assert CHUNK_OVERLAP < CHUNK_LEN - 1

T0 = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def time_is_up():
    return (time.time() - T0) > MAX_SESSION_HOURS * 3600


def find_files(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))


os.makedirs(OUT_DIR, exist_ok=True)
set_seed(SEED)

## 3. Find data, model code and Phase-1 weights

In [ ]:
# ---------- data ----------
if DATA_PKL is None:
    candidates = find_files("tokenized_cjpe.pkl")
    if len(candidates) == 0:
        raise FileNotFoundError("tokenized_cjpe.pkl not found. Attach the output of the CJPE data-prep notebook.")
    candidates.sort(key=lambda p: os.path.getmtime(p))
    DATA_PKL = candidates[-1]
    if len(candidates) > 1:
        print("More than one tokenized_cjpe.pkl found, using the newest:")
        for p in candidates:
            print("   ", p)
print("data:", DATA_PKL)


# ---------- Phase-1 weights ----------
def find_model_dir():
    if MODEL_DIR is not None:
        return MODEL_DIR
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            model_type = json.load(open(cfg_path)).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if model_type == "legal_gqa" and has_weights:
            found.append(folder)
    if len(found) == 0:
        raise FileNotFoundError("No Phase-1 folder with model_type legal_gqa found. Attach the output of your Model 1 (GQA) Phase-1 notebook.")
    final_ones = []
    for d in found:
        if "checkpoint-" not in d:
            final_ones.append(d)
    if len(final_ones) > 0:
        return final_ones[0]
    return max(found, key=lambda d: int(d.rsplit("-", 1)[1]))


MODEL_PATH = find_model_dir()
print("Phase-1 model:", MODEL_PATH)

In [ ]:
# ---------- model code (Model 1) ----------
def find_model_src():
    if MODEL_SRC is not None:
        return MODEL_SRC
    places = ["/kaggle/working/legal_gqa_model.py", os.path.join(MODEL_PATH, "legal_gqa_model.py"),
              os.path.join(os.path.dirname(MODEL_PATH), "legal_gqa_model.py")]
    places = places + find_files("legal_gqa_model.py")
    for p in places:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("legal_gqa_model.py not found. Attach an input that contains it (for example the output of your "
                            "Phase-2 Model-1 notebook), or put it in a small dataset.")


SRC_PATH = find_model_src()
print("model code:", SRC_PATH)
sys.path.insert(0, os.path.dirname(SRC_PATH))
from legal_gqa_model import LegalGQAConfig, LegalGQAForCausalLM
BackboneClass = LegalGQAForCausalLM

In [ ]:
# ---------- resume from an earlier run of this notebook ----------
def copy_missing(src, dst):
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target = os.path.join(dst, rel)
        os.makedirs(target, exist_ok=True)
        for name in files:
            out_path = os.path.join(target, name)
            if not os.path.exists(out_path):
                shutil.copy(os.path.join(root, name), out_path)


resume_src = RESUME_DIR
if resume_src is None:
    prev = find_files(EXPERIMENT_NAME + "/ckpt.pt")
    if len(prev) > 0:
        resume_src = os.path.dirname(prev[-1])
if resume_src is not None and os.path.abspath(resume_src) != os.path.abspath(OUT_DIR):
    copy_missing(resume_src, OUT_DIR)
    print("Resumed files copied from:", resume_src)
else:
    print("No earlier run found, starting fresh.")

## 4. Documents, leak exclusion and pools

In [ ]:
f = open(DATA_PKL, "rb")
bundle = pickle.load(f)
f.close()

SEP_ID = int(bundle["sep_id"])
PAD_ID = int(bundle["pad_id"])
VOCAB_SIZE = int(bundle["vocab_size"])
train_split = bundle["splits"]["train"]
dev_split = bundle["splits"]["dev"]
test_split = bundle["splits"]["test"]
print("train:", len(train_split["doc_ids"]), "| dev:", len(dev_split["doc_ids"]), "| test:", len(test_split["doc_ids"]))
print("SEP:", SEP_ID, "| PAD:", PAD_ID, "| vocab:", VOCAB_SIZE)


def doc_hash(arr):
    return hashlib.md5(np.ascontiguousarray(arr).tobytes()).hexdigest()


# The dev file contains the test documents (with labels). They must never be used for training or validation.
test_hashes = set()
test_id_strings = set()
for i in range(len(test_split["doc_ids"])):
    test_hashes.add(doc_hash(test_split["ids"][i]))
    test_id_strings.add(str(test_split["doc_ids"][i]))


def clean_indices(split):
    keep = []
    removed = 0
    for i in range(len(split["doc_ids"])):
        same_id = str(split["doc_ids"][i]) in test_id_strings
        same_text = doc_hash(split["ids"][i]) in test_hashes
        if same_id or same_text:
            removed += 1
        else:
            keep.append(i)
    return keep, removed


train_keep, train_removed = clean_indices(train_split)
dev_keep, dev_removed = clean_indices(dev_split)
print("Documents that are also in the test set -> excluded. train:", train_removed, "| dev:", dev_removed)
print("Usable: train", len(train_keep), "| dev (validation)", len(dev_keep))


def make_docs(split, indices, with_labels):
    docs = []
    for i in indices:
        d = {"id": split["doc_ids"][i], "ids": split["ids"][i], "label": -1}
        if with_labels:
            d["label"] = int(split["labels"][i])
        docs.append(d)
    return docs


# same random order in every notebook, so pool A / pool B are identical across runs
perm = np.random.RandomState(SEED).permutation(len(train_keep))
pool_a_pos = perm[:N_TRAIN_DOCS]
pool_b_pos = perm[N_TRAIN_DOCS:N_TRAIN_DOCS + N_POOLB_DOCS]
pool_a_idx = []
for p in pool_a_pos:
    pool_a_idx.append(train_keep[int(p)])
pool_b_idx = []
for p in pool_b_pos:
    pool_b_idx.append(train_keep[int(p)])

pool_a = make_docs(train_split, pool_a_idx, True)
pool_b = make_docs(train_split, pool_b_idx, True)
val_docs = make_docs(dev_split, dev_keep, True)
test_docs = make_docs(test_split, list(range(len(test_split["doc_ids"]))), False)
if VAL_MAX_DOCS is not None:
    val_docs = val_docs[:VAL_MAX_DOCS]
if TEST_MAX_DOCS is not None:
    test_docs = test_docs[:TEST_MAX_DOCS]


def label_share(docs):
    if len(docs) == 0:
        return 0.0
    n1 = 0
    for d in docs:
        if d["label"] == 1:
            n1 += 1
    return round(100.0 * n1 / len(docs), 1)


print("pool A:", len(pool_a), "docs, label-1 share", label_share(pool_a), "%")
print("pool B:", len(pool_b), "docs, label-1 share", label_share(pool_b), "%")
print("validation:", len(val_docs), "docs, label-1 share", label_share(val_docs), "%")
print("test:", len(test_docs), "docs")

# class weights from pool A
counts = [0, 0]
for d in pool_a:
    counts[d["label"]] += 1
raw_w = [(1.0 / max(1, counts[0])) ** CLASS_WEIGHT_POWER, (1.0 / max(1, counts[1])) ** CLASS_WEIGHT_POWER]
scale = (counts[0] + counts[1]) / (raw_w[0] * counts[0] + raw_w[1] * counts[1])
CLASS_W = [raw_w[0] * scale, raw_w[1] * scale]
print("pool A counts:", counts, "| class weights:", [round(x, 3) for x in CLASS_W])

## 5. Chunk views (how one judgment becomes model input)

In [ ]:
def get_spans(n_tokens):
    # Returns a list of (start, end) token spans. Each span becomes one chunk of at most CHUNK_LEN tokens (with the SEP added).
    body = CHUNK_LEN - 1
    stride = body - CHUNK_OVERLAP
    if n_tokens <= body:
        return [(0, n_tokens)]
    # windows that end exactly at the end of the judgment, going backwards (the last chunk is always full)
    tail = []
    end = n_tokens
    while True:
        start = max(0, end - body)
        tail.append((start, end))
        if start == 0:
            break
        end = end - stride
    tail.reverse()
    if len(tail) <= MAX_CHUNKS:
        return tail
    if VIEW_MODE == "last":
        return tail[len(tail) - MAX_CHUNKS:]
    if VIEW_MODE == "first":
        spans = []
        s = 0
        for k in range(MAX_CHUNKS):
            spans.append((s, min(n_tokens, s + body)))
            s = s + stride
        return spans
    if VIEW_MODE == "uniform":
        positions = np.linspace(0, len(tail) - 1, MAX_CHUNKS)
        chosen = []
        for p in positions:
            k = int(round(float(p)))
            if k not in chosen:
                chosen.append(k)
        spans = []
        for k in chosen:
            spans.append(tail[k])
        return spans
    # first_last
    spans = []
    s = 0
    for k in range(N_FIRST_CHUNKS):
        spans.append((s, min(n_tokens, s + body)))
        s = s + stride
    keep_tail = MAX_CHUNKS - N_FIRST_CHUNKS
    spans = spans + tail[len(tail) - keep_tail:]
    return spans


def pad_right(seqs, pad_id):
    lengths = []
    for s in seqs:
        lengths.append(len(s))
    max_len = max(lengths)
    input_ids = np.full((len(seqs), max_len), int(pad_id), dtype=np.int64)
    for i in range(len(seqs)):
        input_ids[i, :lengths[i]] = np.asarray(seqs[i], dtype=np.int64)
    return input_ids, np.asarray(lengths, dtype=np.int64)


def make_batch(docs):
    seqs = []
    feats = []
    n_chunks = []
    labels = []
    for d in docs:
        ids = d["ids"]
        n = len(ids)
        spans = get_spans(n)
        for (s, e) in spans:
            piece = ids[s:e].astype(np.int64).tolist()
            piece.append(SEP_ID)
            seqs.append(piece)
            feats.append([s / max(1, n), e / max(1, n), math.log1p(n) / 12.0])
        n_chunks.append(len(spans))
        labels.append(d["label"])
    input_ids, lengths = pad_right(seqs, PAD_ID)
    return {
        "input_ids": input_ids,
        "lengths": lengths,
        "feats": np.asarray(feats, dtype=np.float32),
        "n_chunks": n_chunks,
        "labels": np.asarray(labels, dtype=np.int64),
    }


# quick look at the views on pool A
lens = []
chunk_counts = []
for d in pool_a[:2000]:
    n = len(d["ids"])
    lens.append(n)
    chunk_counts.append(len(get_spans(n)))
print("view mode:", VIEW_MODE, "| chunk_len", CHUNK_LEN, "| overlap", CHUNK_OVERLAP, "| max chunks", MAX_CHUNKS)
print("pool A sample: mean tokens", int(np.mean(lens)), "| mean chunks used", round(float(np.mean(chunk_counts)), 2),
      "| docs cut:", round(100.0 * float(np.mean(np.asarray(lens) > MAX_CHUNKS * (CHUNK_LEN - 1))), 1), "%")

## 6. Adapter layers (written by hand, no peft)

In [ ]:
class LinearWrapper(nn.Module):
    # Common base: keeps the frozen original Linear as self.base and exposes its attributes.
    def __init__(self, base):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.in_features = base.in_features
        self.out_features = base.out_features

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias


class LoRALinear(LinearWrapper):
    # y = W0 x + (alpha / r) * B (A x).  B starts at zero, so the layer starts exactly at the Phase-1 weights.
    def __init__(self, base, r, alpha, dropout):
        super().__init__(base)
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        y = self.base(x)
        z = F.linear(self.drop(x), self.lora_A)
        z = F.linear(z, self.lora_B)
        return y + self.scale * z


_ROT_CACHE = {}


def get_rot_idx(d, device):
    # index table for the row rotation; shared by all layers with the same output size (saves GPU memory)
    key = (d, str(device))
    if key not in _ROT_CACHE:
        ii = torch.arange(d, device=device).unsqueeze(1)
        jj = torch.arange(d, device=device).unsqueeze(0)
        _ROT_CACHE[key] = (jj - ii) % d
    return _ROT_CACHE[key]


class LoRMALinear(LinearWrapper):
    # Low-rank MULTIPLICATIVE update on the output side (Bhardwaj et al., LoRMA):
    #   plus:  W = ((alpha/r) B A + I) W0       B: d_out x r (zeros at start), A: r x d_out
    #   perm:  W = P(B A) W0                    P rotates row i of B A cyclically by i places; starts at P(BA) = I
    def __init__(self, base, r, alpha, dropout, variant):
        super().__init__(base)
        d = base.out_features
        self.d = d
        self.r = r
        self.variant = variant
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lorma_A = nn.Parameter(torch.zeros(r, d))
        self.lorma_B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        else:
            # P(BA) = I at the start: first column of B = ones and A[0, 0] = 1, so BA has ones in column 0 only.
            # "paper": the rest of B is small random, the rest of A is zero.
            # "lora_style": the rest of B is zero, the rest of A is random (like LoRA, gradients flow from the first step).
            with torch.no_grad():
                if PERM_INIT == "lora_style":
                    rest = torch.empty(r - 1, d)
                    nn.init.kaiming_uniform_(rest, a=math.sqrt(5))
                    self.lorma_A.zero_()
                    self.lorma_A[1:, :] = rest
                    self.lorma_B.zero_()
                else:
                    self.lorma_B.normal_(0.0, 0.02)
                    self.lorma_A.zero_()
                self.lorma_B[:, 0] = 1.0
                self.lorma_A[0, 0] = 1.0

    def forward(self, x):
        if self.variant == "plus":
            z = self.base(x)
            return z + self.scale * F.linear(F.linear(self.drop(z), self.lorma_A), self.lorma_B)
        M = self.lorma_B @ self.lorma_A
        R = M.gather(1, get_rot_idx(self.d, M.device))
        w = R.to(self.base.weight.dtype) @ self.base.weight
        b = None
        if self.base.bias is not None:
            b = R.to(self.base.bias.dtype) @ self.base.bias
        return F.linear(self.drop(x), w, b)


class AdapterLinear(LinearWrapper):
    # Bottleneck adapter (Houlsby style) on the output of a Linear:  y = h + Up(GELU(Down(h))),  h = W0 x.
    # Up starts at zero, so the layer starts exactly at the Phase-1 weights.
    def __init__(self, base, bottleneck, dropout):
        super().__init__(base)
        d = base.out_features
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.adapter_down = nn.Linear(d, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, d)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        z = self.adapter_down(self.drop(h))
        z = F.gelu(z)
        z = self.adapter_up(z)
        return h + z


def add_adapters(root, targets, make_layer):
    # Replace every nn.Linear whose attribute name is in `targets` by make_layer(linear).
    replaced = []
    for module_name, module in list(root.named_modules()):
        for child_name, child in list(module.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                setattr(module, child_name, make_layer(child))
                replaced.append(module_name + "." + child_name)
    return replaced


ADAPTER_TAGS = ["lora_A", "lora_B", "lorma_A", "lorma_B", "adapter_down", "adapter_up"]


def is_trainable_name(name):
    if not name.startswith("backbone."):
        return True                      # the new head
    for tag in ADAPTER_TAGS:
        if tag in name:
            return True
    return False

## 7. Model: frozen Phase-1 backbone + adapters + hierarchical head

In [ ]:
def backbone_hidden(b, input_ids, training):
    # Same computation as the Phase-1 forward(), but we stop before lm_head.
    # Right padding needs no mask because the model is causal.
    T = input_ids.shape[1]
    x = b.drop(b.wte(input_ids))
    cos, sin = b._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in b.blocks:
        if training and USE_CHECKPOINT:
            def run(h, blk=blk):
                return blk(h, cos, sin, None, True, None, False)[0]
            x = torch_checkpoint(run, x, use_reentrant=False)
        else:
            x = blk(x, cos, sin, None, True, None, False)[0]
    return b.ln_f(x)


class DocClassifier(nn.Module):
    def __init__(self, backbone, n_classes=2):
        super().__init__()
        self.backbone = backbone
        d = backbone.config.n_embd
        self.proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(HEAD_DROPOUT), nn.Linear(2 * d, HEAD_HIDDEN), nn.GELU())
        self.feat = nn.Linear(3, HEAD_HIDDEN)
        if DOC_HEAD == "bigru":
            self.gru = nn.GRU(HEAD_HIDDEN, HEAD_HIDDEN // 2, batch_first=True, bidirectional=True)
        self.att = nn.Linear(HEAD_HIDDEN, 1)
        self.out = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(HEAD_HIDDEN, n_classes))

    def hidden(self, input_ids):
        return backbone_hidden(self.backbone, input_ids, self.training)

    def chunk_vectors(self, input_ids, lengths):
        # one vector per chunk: mean of the token states + state at the final SEP (the last real token)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            h = self.hidden(input_ids)
        h = h.float()
        N, T, d = h.shape
        positions = torch.arange(T, device=h.device).unsqueeze(0)
        mask = (positions < lengths.unsqueeze(1)).float()
        mean = (h * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(1).float()
        last = h[torch.arange(N, device=h.device), lengths - 1]
        return torch.cat([mean, last], dim=-1)

    def forward(self, batch):
        input_ids = torch.from_numpy(batch["input_ids"]).to(DEVICE)
        lengths = torch.from_numpy(batch["lengths"]).to(DEVICE)
        feats = torch.from_numpy(batch["feats"]).to(DEVICE)
        n_chunks = batch["n_chunks"]
        emb = self.chunk_vectors(input_ids, lengths)
        z = self.proj(emb) + self.feat(feats)
        B = len(n_chunks)
        C = max(n_chunks)
        padded = torch.zeros(B, C, z.shape[1], device=z.device)
        valid = torch.zeros(B, C, dtype=torch.bool, device=z.device)
        pos = 0
        for b in range(B):
            c = n_chunks[b]
            padded[b, :c] = z[pos:pos + c]
            valid[b, :c] = True
            pos = pos + c
        if DOC_HEAD == "bigru":
            packed = nn.utils.rnn.pack_padded_sequence(padded, torch.tensor(n_chunks), batch_first=True, enforce_sorted=False)
            out, _ = self.gru(packed)
            padded, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=C)
        scores = self.att(padded).squeeze(-1)
        scores = scores.masked_fill(~valid, -1e4)
        weights = torch.softmax(scores, dim=1)
        doc_vec = (weights.unsqueeze(-1) * padded).sum(dim=1)
        return self.out(doc_vec)


LOAD_REPORTED = False


def load_backbone():
    global LOAD_REPORTED
    backbone, info = BackboneClass.from_pretrained(MODEL_PATH, output_loading_info=True)
    if not LOAD_REPORTED:
        missing = []
        for k in info.get("missing_keys", []):
            if k != "lm_head.weight":
                missing.append(k)
        unexpected = list(info.get("unexpected_keys", []))
        print("Phase-1 weights loaded | missing keys:", len(missing), "| unexpected keys:", len(unexpected))
        if len(missing) > 0:
            raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing[:5]))
        LOAD_REPORTED = True
    backbone.config.use_cache = False
    backbone.float()
    return backbone


def count_params(model):
    total = 0
    trainable = 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    return total, trainable


def build_model(check_identity=False):
    backbone = load_backbone()
    backbone.to(DEVICE)
    for p in backbone.parameters():
        p.requires_grad = False

    sample = None
    h_before = None
    if check_identity:
        backbone.eval()
        sample_ids = torch.from_numpy(np.asarray(val_docs[0]["ids"][:96].astype(np.int64))).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            h_before = backbone_hidden(backbone, sample_ids, False).float().cpu()
        sample = sample_ids

    if ADAPTER == "lora":
        names = add_adapters(backbone, LORA_TARGETS, lambda lin: LoRALinear(lin, LORA_R, LORA_ALPHA, LORA_DROPOUT))
    elif ADAPTER == "lorma":
        names = add_adapters(backbone, LORMA_TARGETS, lambda lin: LoRMALinear(lin, LORMA_R, LORMA_ALPHA, LORMA_DROPOUT, LORMA_VARIANT))
    else:
        names = add_adapters(backbone, ADAPTER_TARGETS, lambda lin: AdapterLinear(lin, ADAPTER_BOTTLENECK, ADAPTER_DROPOUT))
    model = DocClassifier(backbone)
    model.to(DEVICE)

    if check_identity:
        model.eval()
        with torch.no_grad():
            h_after = model.hidden(sample).float().cpu()
        diff = (h_before - h_after).abs().max().item()
        print("identity check (adapters must not change the Phase-1 output at start): max|diff| = %.2e" % diff)
        if diff > 1e-3:
            raise RuntimeError("Adapters do not start at the identity.")

    for name, p in model.named_parameters():
        p.requires_grad = is_trainable_name(name)
    return model, names


model, replaced_names = build_model(check_identity=True)
TOTAL_PARAMS, TRAINABLE_PARAMS = count_params(model)
print("adapter:", ADAPTER, "| layers wrapped:", len(replaced_names))
print("total %.2fM | trainable %.3fM (%.2f%%)" % (TOTAL_PARAMS / 1e6, TRAINABLE_PARAMS / 1e6, 100.0 * TRAINABLE_PARAMS / TOTAL_PARAMS))

# the Phase-1 weights must be frozen
n_bad = 0
for name, p in model.named_parameters():
    if p.requires_grad and not is_trainable_name(name):
        n_bad += 1
    if (not p.requires_grad) and is_trainable_name(name):
        n_bad += 1
assert n_bad == 0
backbone_trainable = [n for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
for n in backbone_trainable:
    assert any(tag in n for tag in ADAPTER_TAGS)
print("frozen check passed: only adapter and head parameters are trainable")

## 8. Training (resumes by itself)

In [ ]:
def trainable_state(model):
    state = {}
    for name, p in model.named_parameters():
        if p.requires_grad:
            state[name] = p.detach().cpu().clone()
    return state


def load_trainable_state(model, state):
    n_loaded = 0
    for name, p in model.named_parameters():
        if name in state:
            p.data.copy_(state[name].to(p.device))
            n_loaded += 1
    return n_loaded


def macro_f1_binary(y_true, y_pred):
    scores = []
    for c in [0, 1]:
        tp = int(((y_true == c) & (y_pred == c)).sum())
        fp = int(((y_true != c) & (y_pred == c)).sum())
        fn = int(((y_true == c) & (y_pred != c)).sum())
        if tp == 0:
            scores.append(0.0)
        else:
            p = tp / (tp + fp)
            r = tp / (tp + fn)
            scores.append(2 * p * r / (p + r))
    return float(sum(scores) / 2.0)


def best_threshold(prob1, y_true):
    best_t = 0.5
    best_f = -1.0
    for t in np.arange(0.20, 0.801, 0.01):
        f1 = macro_f1_binary(y_true, (prob1 >= t).astype(int))
        if f1 > best_f:
            best_f = f1
            best_t = float(t)
    return best_t, best_f


def predict_logits(model, docs, micro=None):
    if micro is None:
        micro = EVAL_MICRO_DOCS
    model.eval()
    order = np.argsort([len(d["ids"]) for d in docs])
    out = np.zeros((len(docs), 2), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(docs), micro):
            idx = order[start:start + micro]
            batch = make_batch([docs[int(i)] for i in idx])
            logits = model(batch)
            out[idx] = logits.float().cpu().numpy()
    return out


def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def evaluate(model, docs):
    logits = predict_logits(model, docs)
    prob1 = softmax_np(logits)[:, 1]
    y = np.asarray([d["label"] for d in docs])
    f_half = macro_f1_binary(y, (prob1 >= 0.5).astype(int))
    t, f_best = best_threshold(prob1, y)
    return {"f1_at_0.5": f_half, "f1_best": f_best, "threshold": t}, logits


params_adapter = []
params_head = []
for name, p in model.named_parameters():
    if p.requires_grad:
        if name.startswith("backbone."):
            params_adapter.append(p)
        else:
            params_head.append(p)
optimizer = torch.optim.AdamW([
    {"params": params_adapter, "lr": LR_ADAPTER, "weight_decay": 0.0, "base_lr": LR_ADAPTER},
    {"params": params_head, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY, "base_lr": LR_HEAD},
])
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
class_w_tensor = torch.tensor(CLASS_W, dtype=torch.float32, device=DEVICE)

steps_per_epoch = math.ceil(len(pool_a) / BATCH_DOCS)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, int(WARMUP_FRAC * total_steps))
print("steps per epoch:", steps_per_epoch, "| total steps:", total_steps, "| warmup:", warmup_steps)


def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


CKPT_PATH = os.path.join(OUT_DIR, "ckpt.pt")
BEST_PATH = os.path.join(OUT_DIR, "best.pt")
step = 0
best_f1 = -1.0
best_step = 0
bad_evals = 0
history = []
train_seconds = 0.0
tokens_seen = 0

if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    load_trainable_state(model, ck["trainable"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    step = ck["step"]
    best_f1 = ck["best_f1"]
    best_step = ck["best_step"]
    bad_evals = ck["bad_evals"]
    history = ck["history"]
    train_seconds = ck["train_seconds"]
    tokens_seen = ck["tokens_seen"]
    print("RESUMED from step", step, "| best val macro-F1 so far", round(best_f1, 4))
    if step > total_steps:
        raise RuntimeError("The attached checkpoint is at step %d but this run has only %d steps. Did you change EPOCHS / N_TRAIN_DOCS / BATCH_DOCS? "
                           "Remove the old output from the inputs, or restore the old settings." % (step, total_steps))
    if step == total_steps:
        print("Training was already finished in the attached output, only the predictions will be redone.")


def save_ckpt():
    ck = {
        "trainable": trainable_state(model),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step,
        "best_f1": best_f1,
        "best_step": best_step,
        "bad_evals": bad_evals,
        "history": history,
        "train_seconds": train_seconds,
        "tokens_seen": tokens_seen,
    }
    tmp_path = CKPT_PATH + ".tmp"
    torch.save(ck, tmp_path)
    os.replace(tmp_path, CKPT_PATH)


PAUSED = False
EARLY_STOPPED = False
session_start_step = step
session_t0 = time.time()
window_loss = 0.0
window_count = 0
window_tokens = 0
window_t0 = time.time()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

while step < total_steps:
    epoch = step // steps_per_epoch
    within = step % steps_per_epoch
    order = np.random.RandomState(SEED + epoch).permutation(len(pool_a))
    batch_idx = order[within * BATCH_DOCS:(within + 1) * BATCH_DOCS]
    docs = []
    for i in batch_idx:
        docs.append(pool_a[int(i)])

    torch.manual_seed(SEED * 1000003 + step)       # same dropout after a resume
    factor = lr_factor(step)
    for group in optimizer.param_groups:
        group["lr"] = group["base_lr"] * factor

    model.train()
    optimizer.zero_grad(set_to_none=True)
    weight_sum = 0.0
    for d in docs:
        weight_sum += CLASS_W[d["label"]]

    step_loss = 0.0
    for m in range(0, len(docs), MICRO_DOCS):
        micro = docs[m:m + MICRO_DOCS]
        batch = make_batch(micro)
        logits = model(batch)
        y = torch.from_numpy(batch["labels"]).to(DEVICE)
        loss_sum = F.cross_entropy(logits.float(), y, weight=class_w_tensor, reduction="sum", label_smoothing=LABEL_SMOOTHING)
        loss = loss_sum / weight_sum
        scaler.scale(loss).backward()
        step_loss += float(loss.item())
        window_tokens += int(batch["lengths"].sum())

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    step += 1
    window_loss += step_loss
    window_count += 1

    if step % LOG_EVERY_STEPS == 0 or step == total_steps:
        elapsed = time.time() - window_t0
        tok_per_s = window_tokens / max(1e-6, elapsed)
        remaining = (total_steps - step) * (elapsed / max(1, window_count))
        print("step %d/%d | epoch %.2f | loss %.4f | lr x%.3f | %.0f tok/s | ETA %.0f min" % (
            step, total_steps, step / steps_per_epoch, window_loss / max(1, window_count), factor, tok_per_s, remaining / 60.0))
        tokens_seen += window_tokens
        window_loss = 0.0
        window_count = 0
        window_tokens = 0
        window_t0 = time.time()

    if step % EVAL_EVERY_STEPS == 0 or step == total_steps:
        t_eval = time.time()
        metrics, _ = evaluate(model, val_docs)
        metrics["step"] = step
        history.append(metrics)
        print("   VAL step %d | macro-F1 @0.5: %.4f | best-threshold: %.4f (t=%.2f) | eval %.0f s" % (
            step, metrics["f1_at_0.5"], metrics["f1_best"], metrics["threshold"], time.time() - t_eval))
        if metrics["f1_best"] > best_f1:
            best_f1 = metrics["f1_best"]
            best_step = step
            bad_evals = 0
            torch.save({"trainable": trainable_state(model), "step": step, "metrics": metrics}, BEST_PATH)
            print("   new best saved")
        else:
            bad_evals += 1
        train_seconds += time.time() - session_t0
        session_t0 = time.time()
        save_ckpt()
        window_t0 = time.time()
        if bad_evals >= PATIENCE:
            print("Early stop: no improvement for", PATIENCE, "evaluations.")
            EARLY_STOPPED = True
            break
    elif step % SAVE_EVERY_STEPS == 0:
        train_seconds += time.time() - session_t0
        session_t0 = time.time()
        save_ckpt()

    if time_is_up() or (STOP_AFTER_STEPS is not None and (step - session_start_step) >= STOP_AFTER_STEPS):
        if step < total_steps:
            train_seconds += time.time() - session_t0
            session_t0 = time.time()
            save_ckpt()
            PAUSED = True
            print("PAUSED at step", step, "of", total_steps, "- add this notebook's output as an input and run again to continue.")
            break

train_seconds += time.time() - session_t0
tokens_seen += window_tokens
if DEVICE.type == "cuda":
    PEAK_TRAIN_MEM_GB = torch.cuda.max_memory_allocated() / 1e9
else:
    PEAK_TRAIN_MEM_GB = 0.0
print("training loop finished | paused:", PAUSED, "| steps:", step, "| best val macro-F1:", round(best_f1, 4), "at step", best_step)

## 9. Final predictions, threshold, submission

In [ ]:
def write_outputs():
    # --- load the best weights ---
    if os.path.exists(BEST_PATH):
        best = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
        n_loaded = load_trainable_state(model, best["trainable"])
        print("best weights loaded:", n_loaded, "tensors | from step", best["step"])

    # --- validation ---
    t0 = time.time()
    val_logits = predict_logits(model, val_docs)
    val_seconds = time.time() - t0
    val_prob1 = softmax_np(val_logits)[:, 1]
    y_val = np.asarray([d["label"] for d in val_docs])
    thr, f1_thr = best_threshold(val_prob1, y_val)
    pred_val = (val_prob1 >= thr).astype(int)
    f1_half = macro_f1_binary(y_val, (val_prob1 >= 0.5).astype(int))
    try:
        from sklearn.metrics import roc_auc_score
        auc = float(roc_auc_score(y_val, val_prob1))
    except Exception:
        auc = float("nan")

    tp = int(((y_val == 1) & (pred_val == 1)).sum())
    fp = int(((y_val == 0) & (pred_val == 1)).sum())
    fn = int(((y_val == 1) & (pred_val == 0)).sum())
    tn = int(((y_val == 0) & (pred_val == 0)).sum())
    prec1 = tp / max(1, tp + fp)
    rec1 = tp / max(1, tp + fn)
    prec0 = tn / max(1, tn + fn)
    rec0 = tn / max(1, tn + fp)
    print("VALIDATION | macro-F1 @0.5: %.4f | tuned (t=%.2f): %.4f | AUC %.4f" % (f1_half, thr, f1_thr, auc))
    print("   class 1: precision %.3f recall %.3f | class 0: precision %.3f recall %.3f" % (prec1, rec1, prec0, rec0))
    print("   confusion: TP %d FP %d FN %d TN %d" % (tp, fp, fn, tn))

    # --- per-document file for error analysis ---
    lines = ["id,label,prob1,pred,n_tokens,n_chunks"]
    for i in range(len(val_docs)):
        n = len(val_docs[i]["ids"])
        lines.append("%s,%d,%.5f,%d,%d,%d" % (str(val_docs[i]["id"]).replace(",", ";"), val_docs[i]["label"], val_prob1[i], pred_val[i], n, len(get_spans(n))))
    with open(os.path.join(OUT_DIR, "val_predictions.csv"), "w") as fh:
        fh.write("\n".join(lines) + "\n")
    np.save(os.path.join(OUT_DIR, "val_logits.npy"), val_logits)
    with open(os.path.join(OUT_DIR, "val_ids.json"), "w") as fh:
        json.dump([d["id"] for d in val_docs], fh)

    fp_tokens = []
    fn_tokens = []
    for i in range(len(val_docs)):
        n = len(val_docs[i]["ids"])
        if y_val[i] == 0 and pred_val[i] == 1:
            fp_tokens.append(n)
        if y_val[i] == 1 and pred_val[i] == 0:
            fn_tokens.append(n)

    # --- test ---
    t0 = time.time()
    test_logits = predict_logits(model, test_docs)
    test_seconds = time.time() - t0
    test_prob1 = softmax_np(test_logits)[:, 1]
    test_pred = (test_prob1 >= thr).astype(int)
    np.save(os.path.join(OUT_DIR, "test_logits.npy"), test_logits)
    with open(os.path.join(OUT_DIR, "test_ids.json"), "w") as fh:
        json.dump([d["id"] for d in test_docs], fh)

    rows = []
    for i in range(len(test_docs)):
        rows.append({"id": test_docs[i]["id"], "pred": int(test_pred[i])})
    assert len(rows) == len(test_docs)
    for r in rows:
        assert r["pred"] in (0, 1)
    pred_path = os.path.join(OUT_DIR, "prediction.json")
    with open(pred_path, "w") as fh:
        json.dump(rows, fh)
    zip_path = os.path.join(OUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")
    print("TEST | wrote", len(rows), "predictions | share predicted 1:", round(100.0 * float(test_pred.mean()), 1), "%")

    # --- pool B logits (for the KD notebook) ---
    poolb_seconds = 0.0
    if SAVE_POOLB_LOGITS and len(pool_b) > 0:
        t0 = time.time()
        poolb_logits = predict_logits(model, pool_b)
        poolb_seconds = time.time() - t0
        np.save(os.path.join(OUT_DIR, "poolb_logits.npy"), poolb_logits)
        with open(os.path.join(OUT_DIR, "poolb_ids.json"), "w") as fh:
            json.dump([d["id"] for d in pool_b], fh)
        with open(os.path.join(OUT_DIR, "poolb_labels.json"), "w") as fh:
            json.dump([d["label"] for d in pool_b], fh)
        print("pool B logits saved:", poolb_logits.shape)

    peak_eval_mem = 0.0
    if DEVICE.type == "cuda":
        peak_eval_mem = torch.cuda.max_memory_allocated() / 1e9

    results = {
        "experiment": EXPERIMENT_NAME, "model_kind": MODEL_KIND, "adapter": ADAPTER, "smoke_test": SMOKE_TEST,
        "total_params": TOTAL_PARAMS, "trainable_params": TRAINABLE_PARAMS,
        "trainable_pct": 100.0 * TRAINABLE_PARAMS / TOTAL_PARAMS,
        "val_macro_f1_at_0.5": f1_half, "val_macro_f1_tuned": f1_thr, "threshold": thr, "val_auc": auc,
        "val_class1_precision": prec1, "val_class1_recall": rec1, "val_class0_precision": prec0, "val_class0_recall": rec0,
        "val_confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn},
        "val_fp_mean_tokens": float(np.mean(fp_tokens)) if len(fp_tokens) > 0 else None,
        "val_fn_mean_tokens": float(np.mean(fn_tokens)) if len(fn_tokens) > 0 else None,
        "val_docs": len(val_docs), "test_docs": len(test_docs), "pool_a_docs": len(pool_a), "pool_b_docs": len(pool_b),
        "best_step": best_step, "steps_done": step, "total_steps": total_steps, "early_stopped": EARLY_STOPPED,
        "train_hours": train_seconds / 3600.0, "tokens_seen_train": tokens_seen,
        "peak_train_mem_gb": PEAK_TRAIN_MEM_GB, "peak_mem_gb_incl_eval": peak_eval_mem,
        "val_infer_sec_per_doc": val_seconds / max(1, len(val_docs)), "test_infer_sec_per_doc": test_seconds / max(1, len(test_docs)),
        "poolb_infer_seconds": poolb_seconds,
        "settings": {"view_mode": VIEW_MODE, "chunk_len": CHUNK_LEN, "overlap": CHUNK_OVERLAP, "max_chunks": MAX_CHUNKS,
                     "n_first_chunks": N_FIRST_CHUNKS, "doc_head": DOC_HEAD, "epochs": EPOCHS, "batch_docs": BATCH_DOCS,
                     "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD, "class_weights": CLASS_W, "seed": SEED,
                     "n_train_docs": N_TRAIN_DOCS, "n_poolb_docs": N_POOLB_DOCS},
        "history": history,
    }
    with open(os.path.join(OUT_DIR, "results.json"), "w") as fh:
        json.dump(results, fh, indent=1)
    return results


if PAUSED:
    print("Training is paused, so no predictions were written yet. Run again with this output attached.")
    RESULTS = None
else:
    RESULTS = write_outputs()
    print("\nfiles in", OUT_DIR)
    for name in sorted(os.listdir(OUT_DIR)):
        full = os.path.join(OUT_DIR, name)
        if os.path.isfile(full):
            print("  ", name.ljust(22), round(os.path.getsize(full) / 1e6, 2), "MB")

## Notes

- Upload `submission.zip` (it contains `prediction.json`) to Codabench. The threshold was tuned on the validation set (994 clean dev documents in the real data).
- The dev file contains the test documents with their labels. This notebook excludes every train/dev document whose text or id is also in the test set, so validation and training never touch them.
- Pool A (training) and pool B (unseen train documents) come from the same seeded shuffle in every notebook, so `poolb_logits.npy` of different runs line up for knowledge distillation.
- Section ablation: change `VIEW_MODE` (`last`, `first`, `uniform`, `first_last`) and compare `results.json`.
- If the GPU runs out of memory lower `MICRO_DOCS` to 2.